# Pruebas clasificacion de olas

In [0]:
from pyspark.sql import functions as F
env = 'project'

## Seleccionar los datos

In [0]:
data = (
    spark.sql(
        f"""
            SELECT coast_name, datetime, wind_u, wind_v, wave_u, wave_v, wave_period_s
            FROM cor_{env}.silver.swell_metrics
        """
    )
)

In [0]:
data_pre_processing= (
    data
    .withColumn('coast_year_month', F.concat(F.col('coast_name'), F.lit('_'), F.date_format('datetime', 'yyyy-MM')))
)

coast_year_month_dict = {row.coast_year_month: 0.1 for row in data_pre_processing.select('coast_year_month').distinct().collect()}
data_sample = (
    data_pre_processing
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=0)
    .drop('coast_year_month')
).toPandas()

### Validar distribucion de datos

In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [0]:
data_pd = data.toPandas()

## Preparar los datos

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from pyspark.sql import functions as F

In [0]:
features = ['wind_u', 'wind_v', 'wave_u', 'wave_v', 'wave_period_s']

scaler = StandardScaler()
X = scaler.fit_transform(data_sample[features])
X['coast_name'] = data_sample['coast_name']
X['datetime'] = data_sample['datetime']

In [0]:
from sklearn.metrics import silhouette_score as sil_score
import numpy as np
import itertools

In [0]:
posible_e = np.linspace(0.01, 1, 15)